In [ ]:
!pip install praw requests beautifulsoup4 lxml pandas numpy torch transformers tqdm yfinance streamlit plotly python-dotenv -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.3/189.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 46.1 MB/s eta 0:00:00


In [3]:
from data_collector import collect_all, save_raw

TICKER = 'TSLA'
COMPANY = 'Tesla'

raw_df = collect_all(TICKER, company_name=COMPANY)
save_raw(raw_df, TICKER)
raw_df[['source', 'title', 'timestamp']].head(10)


[Yahoo Finance] Fetching headlines for TSLA...
[Yahoo Finance] 20 headlines collected
[Finviz] Scraping news for TSLA...
[Finviz] 100 headlines collected
[Google News] Fetching news for 'TSLA Tesla stock'...
[Google News] 100 articles collected

Total unique headlines: 187
source
google_yahoo_finance                  22
yahoo_finance                         20
finviz_(gurufocus.com)                12
google_marketbeat                     11
google_tipranks                       10
google_the_motley_fool                10
finviz_(barrons.com)                   9
finviz_(yahoo_finance)                 7
google_24/7_wall_st.                   6
finviz_(investor's_business_daily)     6
google_investor's_business_daily       4
finviz_(bloomberg)                     4
finviz_(marketwatch)                   4
finviz_(yahoo_finance_video)           4
finviz_(digitimes)                     4
google_benzinga.com                    3
google_trefis                          3
google_cnbc          

,source,title,timestamp
0,google_tipranks,Soaring Gas Prices Have Customers Reconsiderin...,2026-04-02 17:05:57+00:00
1,yahoo_finance,Lemonade Stock Turned 21% Sweeter Last Month. ...,2026-04-02 16:54:29+00:00
2,yahoo_finance,Stock Market Today: Dow Pares Losses On Hormuz...,2026-04-02 16:47:52+00:00
3,yahoo_finance,Why Tesla Stock Fell After Q1 Deliveries,2026-04-02 16:41:38+00:00
4,yahoo_finance,Tesla Deliveries Rise. Why the Stock Is Falling.,2026-04-02 16:35:00+00:00
5,yahoo_finance,Auto & Transport Roundup: Market Talk,2026-04-02 16:29:00+00:00
6,yahoo_finance,"Tech, Media & Telecom Roundup: Market Talk",2026-04-02 16:28:00+00:00
7,yahoo_finance,Tesla's First-Quarter Deliveries Miss Views as...,2026-04-02 16:25:58+00:00
8,yahoo_finance,Tesla Keeps Momentum Going in China,2026-04-02 16:25:29+00:00
9,yahoo_finance,Walmart Heirs Join the Elite Top 10 Billionair...,2026-04-02 16:25:00+00:00


In [11]:

import pandas as pd
raw_df['score'] = 1

In [12]:
scored_df = score_dataframe(raw_df)
daily_df = aggregate_daily_sentiment(scored_df)
save_sentiment(scored_df, TICKER, level='post')
save_sentiment(daily_df, TICKER, level='daily')
daily_df.tail(10)

[FinBERT] Loading model: ProsusAI/finbert


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[FinBERT] Running on: CUDA


[FinBERT] Scoring batches: 100%|██████████| 6/6 [00:01<00:00,  5.57it/s]


[Scorer] Sentiment distribution:
label
neutral     85
negative    64
positive    38
Name: count, dtype: int64
[Aggregator] Daily sentiment shape: (46, 10)
[Saved] data/sentiment/TSLA_post_sentiment.csv
[Saved] data/sentiment/TSLA_daily_sentiment.csv


,ticker,date,mean_compound,weighted_compound,mean_positive,mean_negative,mean_neutral,sentiment_volume,bull_bear_ratio,dominant_label
36,TSLA,2026-03-19,0.0620,0.0620,0.0898,0.0280,0.8822,2,0.0000,neutral
37,TSLA,2026-03-20,-0.2544,-0.2544,0.0867,0.3411,0.5722,3,0.0000,neutral
38,TSLA,2026-03-24,-0.0201,-0.0201,0.4664,0.4864,0.0472,2,0.5000,negative
39,TSLA,2026-03-27,-0.3773,-0.3773,0.1007,0.4780,0.4213,19,0.0909,negative
40,TSLA,2026-03-28,-0.1796,-0.1796,0.1658,0.3454,0.4888,4,0.5000,neutral
41,TSLA,2026-03-29,-0.3656,-0.3656,0.1248,0.4905,0.3847,4,0.0000,negative
42,TSLA,2026-03-30,-0.0625,-0.0625,0.2723,0.3349,0.3928,13,0.3750,negative
43,TSLA,2026-03-31,0.1505,0.1505,0.3623,0.2119,0.4258,26,0.6154,neutral
44,TSLA,2026-04-01,0.0341,0.0341,0.2409,0.2068,0.5523,36,0.4706,neutral
45,TSLA,2026-04-02,-0.3108,-0.3108,0.1592,0.4700,0.3708,36,0.2609,negative


In [5]:
with open('sentiment_engine.py', 'r') as f:
    print(f.read())

"""
Stage 2: Sentiment Analysis with FinBERT
Runs ProsusAI/finbert on scraped text and produces daily aggregated sentiment scores.
FinBERT is fine-tuned on financial text — much better than VADER/TextBlob for this.
"""

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F
from tqdm import tqdm
import os


# ─────────────────────────────────────────────
# MODEL LOADER
# ─────────────────────────────────────────────

MODEL_NAME = "ProsusAI/finbert"

def load_finbert():
    """
    Downloads and caches FinBERT on first run (~440MB).
    Subsequent runs load from cache — fast.
    """
    print(f"[FinBERT] Loading model: {MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
    model.eval()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    print(f"[F

In [13]:
from backtest_engine import fetch_prices, build_signal_df, run_backtest
prices = fetch_prices(TICKER)
signal_df = build_signal_df(daily_df, prices)
result = run_backtest(signal_df, threshold=0.05)
for k, v in result['metrics'].items():
    print(f'{k}: {v}')

[Prices] Fetching TSLA from 2026-01-02 to 2026-04-02...


/content/backtest_engine.py:31: FutureWarning: YF.download() has changed argument auto_adjust default to True
  raw = yf.download(ticker, start=start, end=end, progress=False)


[Prices] 62 trading days loaded.
[Signal] Merged signal dataframe: 20 rows
ticker: TSLA
n_days: 20
n_trades: 5
trade_pct: 25.0
strategy_total_return: -1.71
bh_total_return: -3.11
strategy_ann_return: -19.56
bh_ann_return: -32.81
strategy_sharpe: -1.653
bh_sharpe: -0.762
strategy_max_drawdown: -3.34
bh_max_drawdown: -10.04
win_rate: 50.0
directional_accuracy: 45.0
sentiment_price_correlation: 0.1252
sentiment_threshold: 0.05


In [14]:
from backtest_engine import sweep_thresholds
sweep = sweep_thresholds(signal_df)
sweep[['sentiment_threshold',
       'strategy_total_return',
       'bh_total_return',
       'strategy_sharpe',
       'directional_accuracy']]

,sentiment_threshold,strategy_total_return,bh_total_return,strategy_sharpe,directional_accuracy
0,-0.20,6.79,-3.11,2.549,45.0
1,-0.15,6.79,-3.11,2.549,45.0
2,-0.10,6.79,-3.11,2.549,45.0
3,-0.05,2.06,-3.11,0.979,45.0
4,-0.00,2.30,-3.11,1.574,45.0
5,0.05,-1.71,-3.11,-1.653,45.0
6,0.10,0.07,-3.11,1.323,45.0
7,0.15,0.07,-3.11,1.323,45.0
8,0.20,0.07,-3.11,1.323,45.0
9,0.25,0.07,-3.11,1.323,45.0


In [16]:

import subprocess
import threading
import time

def run_app():
    subprocess.Popen([
        'streamlit', 'run', 'app.py',
        '--server.port', '8501',
        '--server.headless', 'true',
        '--server.enableCORS', 'false',
        '--server.enableXsrfProtection', 'false'
    ])

t = threading.Thread(target=run_app, daemon=True)
t.start()
time.sleep(4)


from google.colab.output import eval_js
print('App running at:')
print(eval_js("google.colab.kernel.proxyPort(8501)"))

App running at:
https://8501-gpu-t4-s-kkb-usw1b1-1bh4jmsai1r1m-b.us-west1-1.prod.colab.dev


In [17]:
import subprocess
result = subprocess.run(['curl', 'http://localhost:8501'], capture_output=True, text=True)
print(result.stdout[:500])

<!--
 Copyright (c) Streamlit Inc. (2018-2022) Snowflake Inc. (2022-2026)

 Licensed under the Apache License, Version 2.0 (the "License");
 you may not use this file except in compliance with the License.
 You may obtain a copy of the License at

     http://www.apache.org/licenses/LICENSE-2.0

 Unless required by applicable law or agreed to in writing, software
 distributed under the License is distributed on an "AS IS" BASIS,
 WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or im


In [18]:
from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8501)")
print(url)

https://8501-gpu-t4-s-kkb-usw1b1-1bh4jmsai1r1m-b.us-west1-1.prod.colab.dev
